# ISAS Keypoint TCN: train, export `.pt`, and test unseen data

Notebook này thực hiện đúng workflow challenge: clean 8 nhãn, LOSO theo participant, train final trên participant 1,2,3,5, xuất checkpoint `.pt`, rồi dự đoán CSV của participant hoàn toàn mới. `None` không phải target class; `Throwing` được gộp vào `Throwing things`.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import torch

PROJECT_DIR = Path(r'C:\Users\Tan\Documents\keypoint label')
DATA_DIR = Path(r'E:\DATA\Keypoint\Train Data-20260812T151048Z-1-001\Train Data\keypointlabel')
MODEL_PATH = PROJECT_DIR / 'artifacts' / 'keypoint_tcn_1_2_3_5.pt'
METRICS_DIR = PROJECT_DIR / 'artifacts' / 'metrics'
SUBJECTS = [1, 2, 3, 5]

print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Training data:', DATA_DIR)

## 1. Kiểm tra dữ liệu

Cell này chỉ đọc schema và thống kê nhãn; chưa train model.

In [ ]:
for subject in SUBJECTS:
    path = DATA_DIR / f'keypoints_with_labels_{subject}.csv'
    frame = pd.read_csv(path)
    print(f'\nSubject {subject}: {frame.shape}')
    print(frame['Action Label'].fillna('None').value_counts().to_string())

## 2. LOSO và xuất model `.pt`

Mặc định chạy bốn fold LOSO rồi train final trên cả 1,2,3,5. Với RTX 4050, bước này vẫn có thể mất vài phút. Để chỉ kiểm tra pipeline nhanh, đặt `QUICK_RUN=True`; model quick-run không dùng để báo cáo kết quả.

In [ ]:
QUICK_RUN = False

command = [
    sys.executable,
    str(PROJECT_DIR / 'train_export_pt.py'),
    '--data-dir', str(DATA_DIR),
    '--output', str(MODEL_PATH),
    '--metrics-dir', str(METRICS_DIR),
    '--subjects', *map(str, SUBJECTS),
]
if QUICK_RUN:
    command += ['--skip-loso', '--epochs', '1']

result = subprocess.run(command, cwd=PROJECT_DIR, text=True)
if result.returncode != 0:
    raise RuntimeError(f'Training failed with exit code {result.returncode}')
print('Checkpoint:', MODEL_PATH)

## 3. Xác nhận checkpoint load lại độc lập

Checkpoint chứa trọng số, thứ tự 8 class, cấu hình TCN, thông số normalize, mean/std và metadata huấn luyện.

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=True)
print('Architecture:', checkpoint['architecture'])
print('Classes:', checkpoint['classes'])
print('Preprocessing:', json.dumps(checkpoint['preprocessing'], indent=2))
print('Training metadata:', json.dumps(checkpoint['training_metadata'], indent=2))
print(f'Model size: {MODEL_PATH.stat().st_size / 1024 / 1024:.2f} MiB')

## 4. Test trên dataset hoàn toàn mới

Khi có file test, điền `NEW_TEST_CSV`. Output mặc định có đúng ba cột `participant_id,timestamp,predicted_label`. Nếu file test có `Action Label`, notebook tự in accuracy, macro-F1 và abnormal-F1.

In [ ]:
NEW_TEST_CSV = None  # Ví dụ: Path(r'E:\DATA\Keypoint\new_unseen_test.csv')
PARTICIPANT_ID = 'unseen_participant'
SUBMISSION_PATH = PROJECT_DIR / 'submission_predictions.csv'
GROUP_COLUMN = None  # Đặt thành 'sample_id' nếu test đã pre-segmented

if NEW_TEST_CSV is None:
    print('Chưa cấu hình NEW_TEST_CSV. Khi có test data, thay đường dẫn và chạy lại cell này.')
else:
    command = [
        sys.executable,
        str(PROJECT_DIR / 'predict_new_dataset.py'),
        '--model', str(MODEL_PATH),
        '--input', str(NEW_TEST_CSV),
        '--output', str(SUBMISSION_PATH),
        '--participant-id', str(PARTICIPANT_ID),
    ]
    if GROUP_COLUMN:
        command += ['--group-column', GROUP_COLUMN]
    result = subprocess.run(command, cwd=PROJECT_DIR, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Inference failed with exit code {result.returncode}')
    display(pd.read_csv(SUBMISSION_PATH).head())

## 5. Chuẩn bị cho ESP32

File `.pt` không chạy trực tiếp trên ESP32. Sau khi accuracy/F1 trên test mới đạt yêu cầu, bước deployment sẽ là: freeze preprocessing, chuyển model sang int8 TFLite Micro, xuất model thành C array, rồi viết ring buffer 150 frame và inference code trên ESP32/ESP32-S3. Không nên chuyển embedded trước khi xác nhận output của cell 4 vì thay preprocessing sau đó sẽ làm firmware và model không còn khớp nhau.